# Notebook 2 — Inferential Statistics
**Real-Time E-commerce Price Intelligence Platform**

Covers: t-test · Mann-Whitney U · one-way ANOVA · Kruskal-Wallis · linear regression (price ~ rating + time) · confidence intervals · p-values · power analysis

---

In [ ]:
# papermill parameters
run_date = "2024-01-01"
alpha = 0.05
output_html = "reports/02_inferential_stats.html"

In [ ]:
import os
import sys
import warnings
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import (
    ttest_ind, mannwhitneyu, f_oneway, kruskal,
    shapiro, levene, norm,
)
from scipy.stats import t as t_dist

warnings.filterwarnings('ignore')

NB_DIR = Path().resolve()
ROOT = NB_DIR.parent.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(ROOT))

REPORTS_DIR = ROOT / 'analytics' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Run date  : {run_date}')
print(f'Alpha     : {alpha}')
print(f'Root      : {ROOT}')

## 1. Data Loading

In [ ]:
def _load_jsonl() -> pd.DataFrame:
    frames = []
    for jsonl in (ROOT / 'data').rglob('*.jsonl'):
        try:
            frames.append(pd.read_json(jsonl, lines=True))
        except Exception:
            pass
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _synthetic() -> pd.DataFrame:
    rng = np.random.default_rng(42)
    sources = {
        'books_toscrape': dict(mu=15,  sigma=8,  cats=['Fiction','Mystery','Science','History','Children']),
        'scrapeme_live':  dict(mu=30,  sigma=12, cats=['Electronics','Toys','Clothing','Sports']),
        'jumia_ma':       dict(mu=250, sigma=80, cats=['Phones','Computers','Home','Fashion','Beauty']),
        'ultrapc_ma':     dict(mu=450, sigma=150,cats=['Laptops','Desktops','Monitors','Accessories']),
        'micromagma_ma':  dict(mu=380, sigma=120,cats=['Laptops','Desktops','Peripherals','Networking']),
    }
    rows = []
    pid = 1
    for i in range(60):
        for day_offset in range(30):
            for src, cfg in sources.items():
                cat = rng.choice(cfg['cats'])
                rows.append({
                    'product_id': f'p{pid:04d}',
                    'source': src,
                    'category': cat,
                    'title': f'{cat} product {pid}',
                    'price': round(abs(rng.normal(cfg['mu'], cfg['sigma'])), 2),
                    'currency': 'MAD' if 'ma' in src else 'GBP',
                    'rating': round(rng.uniform(1, 5), 1),
                    'availability': rng.choice(['In stock', 'Out of stock'], p=[0.85, 0.15]),
                    'scraped_at': pd.Timestamp(run_date) + pd.Timedelta(days=day_offset, hours=int(rng.integers(0, 23))),
                })
        pid += 1
    return pd.DataFrame(rows)


df = pd.DataFrame()
try:
    from bigtable.client import BigtableClient
    client = BigtableClient(
        project_id=os.environ.get('GCP_PROJECT_ID', 'local'),
        instance_id=os.environ.get('BIGTABLE_INSTANCE_ID', 'price-intelligence'),
    )
    rows_bt = list(client._table.read_rows(limit=10000))
    if rows_bt:
        df = pd.DataFrame([client._row_to_dict(r) for r in rows_bt])
        print(f'[Bigtable] {len(df):,} rows')
except Exception as e:
    print(f'Bigtable unavailable: {e}')

if df.empty:
    df = _load_jsonl()
    if not df.empty:
        print(f'[JSONL] {len(df):,} rows')

if df.empty:
    df = _synthetic()
    print(f'[Synthetic demo] {len(df):,} rows')

df['price'] = pd.to_numeric(df.get('price'), errors='coerce')
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]
if 'scraped_at' in df.columns:
    df['scraped_at'] = pd.to_datetime(df['scraped_at'], errors='coerce', utc=True)
    df['days_since_start'] = (
        df['scraped_at'] - df['scraped_at'].min()
    ).dt.total_seconds() / 86400
if 'category' not in df.columns:
    df['category'] = 'Unknown'
if 'rating' not in df.columns:
    rng2 = np.random.default_rng(99)
    df['rating'] = np.round(rng2.uniform(1, 5, len(df)), 1)

sources = df['source'].unique().tolist()
print(f'Dataset: {len(df):,} rows, {len(sources)} sources')
print(f'Sources: {sources}')

## 2. Normality Check — Shapiro-Wilk
Determines whether parametric or non-parametric tests are appropriate

In [ ]:
normality_rows = []
for src in sources:
    s = df[df['source'] == src]['price'].dropna()
    sample = s.sample(min(len(s), 5000), random_state=42)
    stat, p = shapiro(sample)
    normality_rows.append({
        'source': src,
        'n': len(s),
        'W': round(stat, 4),
        'p_value': round(p, 6),
        'normal': 'Yes' if p >= alpha else 'No',
    })

normality_df = pd.DataFrame(normality_rows)
print('=== Shapiro-Wilk Normality Test ===')
print(normality_df.to_string(index=False))
print()
n_normal = normality_df['normal'].eq('Yes').sum()
print(f'{n_normal}/{len(sources)} sources pass normality test')
print('Recommendation: use non-parametric tests (Mann-Whitney / Kruskal-Wallis) as primary')

## 3. Levene's Test — Homogeneity of Variance

In [ ]:
groups_all = [df[df['source'] == s]['price'].dropna().values for s in sources]
lev_stat, lev_p = levene(*groups_all)
print(f"Levene's test: F={lev_stat:.4f}, p={lev_p:.4e}")
print(f"Variances are {'NOT ' if lev_p < alpha else ''}homogeneous at α={alpha}")
print('→ Use Welch t-test (equal_var=False) to be safe')

## 4. Welch t-test — Pairwise Mean Comparison

In [ ]:
ttest_rows = []
for src_a, src_b in itertools.combinations(sources, 2):
    a = df[df['source'] == src_a]['price'].dropna()
    b = df[df['source'] == src_b]['price'].dropna()
    stat, p = ttest_ind(a, b, equal_var=False)
    ttest_rows.append({
        'source_A': src_a,
        'source_B': src_b,
        'mean_A': round(a.mean(), 2),
        'mean_B': round(b.mean(), 2),
        'mean_diff': round(a.mean() - b.mean(), 2),
        't_stat': round(stat, 4),
        'p_value': round(p, 6),
        'significant': 'Yes' if p < alpha else 'No',
    })

ttest_df = pd.DataFrame(ttest_rows).sort_values('p_value')
print(f'=== Welch t-test (α={alpha}) ===')
print(ttest_df.to_string(index=False))
print(f'\n{ttest_df["significant"].eq("Yes").sum()}/{len(ttest_df)} pairs significantly different')

## 5. Mann-Whitney U Test — Non-parametric Pairwise
No normality assumption required

In [ ]:
mwu_rows = []
for src_a, src_b in itertools.combinations(sources, 2):
    a = df[df['source'] == src_a]['price'].dropna()
    b = df[df['source'] == src_b]['price'].dropna()
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    n1, n2 = len(a), len(b)
    r = 1 - (2 * stat) / (n1 * n2)  # rank-biserial correlation
    mwu_rows.append({
        'source_A': src_a,
        'source_B': src_b,
        'median_A': round(a.median(), 2),
        'median_B': round(b.median(), 2),
        'U_stat': round(stat, 0),
        'p_value': round(p, 6),
        'effect_r': round(r, 3),
        'significant': 'Yes' if p < alpha else 'No',
    })

mwu_df = pd.DataFrame(mwu_rows).sort_values('p_value')
print(f'=== Mann-Whitney U (α={alpha}) ===')
print(mwu_df.to_string(index=False))
print('\nEffect size |r|: small≥0.1, medium≥0.3, large≥0.5')

# p-value heatmap
pivot_p = mwu_df.pivot_table(index='source_A', columns='source_B', values='p_value')
fig_mwu = go.Figure(data=go.Heatmap(
    z=np.log10(pivot_p.values + 1e-10),
    x=pivot_p.columns.tolist(),
    y=pivot_p.index.tolist(),
    colorscale='RdBu',
    hovertemplate='%{y} vs %{x}<br>log10(p)=%{z:.2f}<extra></extra>',
))
fig_mwu.update_layout(
    title='Mann-Whitney U — log10(p-value) (more negative = more significant)',
    xaxis_tickangle=-35,
)
fig_mwu.show()

## 6. One-Way ANOVA
H₀: μ₁ = μ₂ = ... = μₖ (all source means equal)

In [ ]:
anova_stat, anova_p = f_oneway(*groups_all)

grand_mean = df['price'].mean()
ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups_all)
ss_total   = ((df['price'] - grand_mean)**2).sum()
eta_sq = round(ss_between / ss_total, 4)

print('=== One-Way ANOVA ===')
print(f'  F-statistic : {anova_stat:.4f}')
print(f'  p-value     : {anova_p:.4e}')
print(f'  eta-squared : {eta_sq}  (small≥.01, medium≥.06, large≥.14)')
if anova_p < alpha:
    print(f'→ REJECT H₀: at least one source has a different mean price')
else:
    print(f'→ FAIL TO REJECT H₀')

## 7. Kruskal-Wallis Test
Non-parametric ANOVA — rank-based, no normality assumption

In [ ]:
kw_stat, kw_p = kruskal(*groups_all)

n_total = sum(len(g) for g in groups_all)
eps_sq  = round((kw_stat - len(groups_all) + 1) / (n_total - len(groups_all)), 4)

print('=== Kruskal-Wallis Test ===')
print(f'  H-statistic : {kw_stat:.4f}')
print(f'  p-value     : {kw_p:.4e}')
print(f'  epsilon-sq  : {eps_sq}')
if kw_p < alpha:
    print(f'→ REJECT H₀: price distributions differ across sources')
else:
    print(f'→ FAIL TO REJECT H₀')

fig_kw = px.box(
    df, x='source', y='price', color='source',
    log_y=True, points='outliers',
    title=f'Price by Source — Kruskal-Wallis H={kw_stat:.2f}, p={kw_p:.2e}',
)
fig_kw.show()

## 8. ANOVA by Category

In [ ]:
cats = df['category'].value_counts().head(8).index.tolist()
cat_groups = [df[df['category'] == c]['price'].dropna().values for c in cats if len(df[df['category'] == c]) >= 5]

if len(cat_groups) >= 2:
    cat_f, cat_p   = f_oneway(*cat_groups)
    kw_cat_h, kw_cat_p = kruskal(*cat_groups)
    print('=== ANOVA & Kruskal-Wallis — Price by Category ===')
    print(f'  ANOVA F={cat_f:.4f}, p={cat_p:.4e}')
    print(f'  K-W   H={kw_cat_h:.4f}, p={kw_cat_p:.4e}')

    cat_means = pd.DataFrame({
        'category': [c for c in cats if len(df[df['category'] == c]) >= 5],
        'n': [len(g) for g in cat_groups],
        'mean':   [round(g.mean(), 2) for g in cat_groups],
        'median': [round(float(np.median(g)), 2) for g in cat_groups],
    }).sort_values('median', ascending=False)
    print()
    print(cat_means.to_string(index=False))

    fig_cat = px.box(
        df[df['category'].isin(cats)],
        x='category', y='price', color='category',
        log_y=True, title=f'Price by Category — ANOVA p={cat_p:.2e}',
    )
    fig_cat.update_layout(xaxis_tickangle=-30, showlegend=False)
    fig_cat.show()
else:
    print('Not enough category groups')

## 9. Confidence Intervals (95%) — Mean Price per Source

In [ ]:
ci_rows = []
for src in sources:
    s = df[df['source'] == src]['price'].dropna()
    n    = len(s)
    mean = s.mean()
    se   = s.sem()
    lo, hi = t_dist.interval(1 - alpha, df=n - 1, loc=mean, scale=se)
    ci_rows.append({
        'source': src,
        'n': n,
        'mean': round(mean, 2),
        'se': round(se, 3),
        'ci_lower': round(lo, 2),
        'ci_upper': round(hi, 2),
        'ci_width': round(hi - lo, 2),
    })

ci_df = pd.DataFrame(ci_rows)
print('=== 95% Confidence Intervals for Mean Price ===')
print(ci_df.to_string(index=False))

fig_ci = go.Figure()
for _, row in ci_df.iterrows():
    fig_ci.add_trace(go.Scatter(
        x=[row['ci_lower'], row['ci_upper']],
        y=[row['source'], row['source']],
        mode='lines', line=dict(width=5), showlegend=False,
    ))
    fig_ci.add_trace(go.Scatter(
        x=[row['mean']], y=[row['source']],
        mode='markers', marker=dict(size=12, symbol='diamond'), showlegend=False,
    ))
fig_ci.update_layout(
    title='95% Confidence Intervals for Mean Price per Source',
    xaxis_title='Price', yaxis_title='Source', height=350,
)
fig_ci.show()

## 10. Linear Regression — price ~ rating

In [ ]:
reg_df = df.dropna(subset=['price', 'rating'])

reg_rows = []
for src in sources:
    s = reg_df[reg_df['source'] == src]
    if len(s) < 10:
        continue
    slope, intercept, r, p, se = stats.linregress(s['rating'], s['price'])
    reg_rows.append({
        'source': src,
        'n': len(s),
        'slope': round(slope, 4),
        'intercept': round(intercept, 2),
        'r': round(r, 4),
        'r2': round(r**2, 4),
        'p_value': round(p, 6),
        'se': round(se, 4),
        'significant': 'Yes' if p < alpha else 'No',
    })

reg_rating_df = pd.DataFrame(reg_rows)
print('=== Regression: price ~ rating (per source) ===')
print(reg_rating_df.to_string(index=False))

fig_reg = px.scatter(
    reg_df, x='rating', y='price', color='source',
    opacity=0.4, trendline='ols', log_y=True,
    title='Price vs Rating — OLS trendlines (log y)',
    labels={'price': 'Price (log)', 'rating': 'Rating'},
)
fig_reg.show()

## 11. Linear Regression — price ~ time

In [ ]:
if 'days_since_start' in df.columns:
    time_df  = df.dropna(subset=['price', 'days_since_start'])
    time_rows = []
    for src in sources:
        s = time_df[time_df['source'] == src]
        if len(s) < 10:
            continue
        slope, intercept, r, p, se = stats.linregress(s['days_since_start'], s['price'])
        time_rows.append({
            'source': src,
            'slope_per_day': round(slope, 4),
            'r2': round(r**2, 4),
            'p_value': round(p, 6),
            'trend': 'Rising' if slope > 0 else 'Falling',
        })
    time_reg_df = pd.DataFrame(time_rows)
    print('=== Regression: price ~ time (per source) ===')
    print(time_reg_df.to_string(index=False))

    fig_time = px.scatter(
        time_df, x='days_since_start', y='price', color='source',
        opacity=0.3, trendline='ols', log_y=True,
        title='Price vs Time — OLS trendlines',
        labels={'price': 'Price (log)', 'days_since_start': 'Days since start'},
    )
    fig_time.show()
else:
    time_reg_df = pd.DataFrame()
    print('No scraped_at column — skipping time regression')

## 12. Multiple Regression — log(price) ~ rating + days
Using statsmodels for full OLS summary

In [ ]:
ols_summary_html = '<p>statsmodels not installed</p>'
try:
    import statsmodels.formula.api as smf

    model_df = df.dropna(subset=['price', 'rating']).copy()
    model_df['log_price'] = np.log(model_df['price'])

    formula_parts = ['rating']
    if 'days_since_start' in model_df.columns:
        model_df = model_df.dropna(subset=['days_since_start'])
        formula_parts.append('days_since_start')

    formula = 'log_price ~ ' + ' + '.join(formula_parts)
    model   = smf.ols(formula=formula, data=model_df).fit()

    print(f'=== Multiple Regression: {formula} ===')
    print(model.summary())
    ols_summary_html = model.summary().as_html()
except ImportError:
    print('statsmodels not available')

## 13. Power Analysis
Minimum sample size to detect effects at 80% / 90% / 95% power

In [ ]:
def required_n(d: float, alpha: float = 0.05, power: float = 0.80) -> int:
    z_a = norm.ppf(1 - alpha / 2)
    z_b = norm.ppf(power)
    return int(np.ceil(2 * ((z_a + z_b) / d) ** 2))


power_rows = []
for label, d in [('Small (d=0.2)', 0.2), ('Medium (d=0.5)', 0.5), ('Large (d=0.8)', 0.8)]:
    for pw in [0.80, 0.90, 0.95]:
        power_rows.append({
            'effect_size': label,
            'power': f'{int(pw*100)}%',
            'n_per_group': required_n(d, alpha=alpha, power=pw),
        })

power_df = pd.DataFrame(power_rows)
print(f'=== Power Analysis (α={alpha}) ===')
print(power_df.to_string(index=False))

print('\n=== Actual n vs required (medium effect, 80% power) ===')
req = required_n(0.5)
for src in sources:
    n = len(df[df['source'] == src])
    adequacy = 'OK' if n >= req else 'UNDERPOWERED'
    print(f'  {src:25s}: n={n:,}  (required={req})  {adequacy}')

In [ ]:
ns = np.arange(5, 600, 5)

def compute_power(n_arr, d=0.5, a=0.05):
    z_alpha = norm.ppf(1 - a / 2)
    ncp = d * np.sqrt(n_arr / 2)
    return 1 - norm.cdf(z_alpha - ncp) + norm.cdf(-z_alpha - ncp)

fig_pwr = go.Figure()
for d_val, label in [(0.2, 'Small d=0.2'), (0.5, 'Medium d=0.5'), (0.8, 'Large d=0.8')]:
    fig_pwr.add_trace(go.Scatter(
        x=ns, y=compute_power(ns, d=d_val),
        name=label, mode='lines',
    ))
fig_pwr.add_hline(y=0.80, line_dash='dash', line_color='red', annotation_text='80%')
fig_pwr.add_hline(y=0.95, line_dash='dot', line_color='orange', annotation_text='95%')
fig_pwr.update_layout(
    title=f'Statistical Power Curves (α={alpha})',
    xaxis_title='n per group', yaxis_title='Power',
    yaxis_range=[0, 1.05],
)
fig_pwr.show()

## 14. Effect Sizes — Cohen's d Between Source Pairs

In [ ]:
def cohens_d(a, b):
    pooled_sd = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return (a.mean() - b.mean()) / pooled_sd if pooled_sd > 0 else 0.0


effect_rows = []
for src_a, src_b in itertools.combinations(sources, 2):
    a = df[df['source'] == src_a]['price'].dropna()
    b = df[df['source'] == src_b]['price'].dropna()
    d = cohens_d(a, b)
    effect_rows.append({
        'comparison': f'{src_a} vs {src_b}',
        "Cohen_d": round(d, 3),
        'magnitude': 'large' if abs(d) >= 0.8 else ('medium' if abs(d) >= 0.5 else 'small'),
    })

effect_df = pd.DataFrame(effect_rows).sort_values('Cohen_d', key=abs, ascending=False)
print("=== Cohen's d Effect Sizes ===")
print(effect_df.to_string(index=False))

## 15. Save HTML Report

In [ ]:
html_path = REPORTS_DIR / '02_inferential_stats.html'

time_reg_html = time_reg_df.to_html(index=False) if not time_reg_df.empty else '<p>No time data available</p>'

html = f"""<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='UTF-8'/>
  <title>Inferential Statistics Report — {run_date}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; line-height: 1.6; }}
    h1   {{ color: #1a1a2e; }}
    h2   {{ color: #16213e; border-bottom: 2px solid #0f3460; padding-bottom:4px; margin-top:32px; }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 24px; font-size: .9em; }}
    th   {{ background: #0f3460; color: white; padding: 8px 12px; text-align:left; }}
    td   {{ padding: 6px 12px; border-bottom: 1px solid #ddd; }}
    tr:nth-child(even) {{ background: #f8f9fa; }}
    .meta   {{ color: #666; font-size: .9em; }}
    .verdict {{ padding: 10px 16px; border-left: 4px solid #0f3460; background: #f0f4ff; margin: 8px 0; }}
  </style>
</head>
<body>
<h1>Inferential Statistics Report</h1>
<p class='meta'>Run date: {run_date} | Records: {len(df):,} | &alpha; = {alpha} | Sources: {len(sources)}</p>

<h2>1. Normality (Shapiro-Wilk)</h2>
{normality_df.to_html(index=False)}

<h2>2. Homogeneity of Variance (Levene)</h2>
<div class='verdict'>F = {lev_stat:.4f}, p = {lev_p:.4e} &mdash;
Variances {'<b>NOT</b> homogeneous' if lev_p < alpha else '<b>homogeneous</b>'} at &alpha;={alpha}</div>

<h2>3. Welch t-test (pairwise)</h2>
{ttest_df.to_html(index=False)}

<h2>4. Mann-Whitney U (non-parametric)</h2>
{mwu_df.to_html(index=False)}

<h2>5. One-Way ANOVA</h2>
<div class='verdict'>F = {anova_stat:.4f}, p = {anova_p:.4e}, &eta;&sup2; = {eta_sq}<br/>
{'&rarr; REJECT H&sub0;: significant price differences across sources' if anova_p < alpha else '&rarr; FAIL TO REJECT H&sub0;'}</div>

<h2>6. Kruskal-Wallis</h2>
<div class='verdict'>H = {kw_stat:.4f}, p = {kw_p:.4e}, &epsilon;&sup2; = {eps_sq}<br/>
{'&rarr; REJECT H&sub0;' if kw_p < alpha else '&rarr; FAIL TO REJECT H&sub0;'}</div>

<h2>7. Confidence Intervals (95%)</h2>
{ci_df.to_html(index=False)}

<h2>8. Regression: price &sim; rating</h2>
{reg_rating_df.to_html(index=False)}

<h2>9. Regression: price &sim; time</h2>
{time_reg_html}

<h2>10. OLS Multiple Regression Summary</h2>
{ols_summary_html}

<h2>11. Cohen's d Effect Sizes</h2>
{effect_df.to_html(index=False)}

<h2>12. Power Analysis</h2>
{power_df.to_html(index=False)}
</body></html>"""

html_path.write_text(html, encoding='utf-8')
print(f'HTML report saved → {html_path}')
print('\n✓ Notebook 2 — Inferential Statistics complete')